In [0]:
# Cell 1 — Setup
import numpy as np
import pandas as pd
import mlflow
import dowhy
from dowhy import CausalModel
import warnings
warnings.filterwarnings('ignore')

gold_df = spark.table("causal_project.gold_household").toPandas()

# Same encoding as modeling notebook
gold_encoded = gold_df.copy()
gold_encoded['age_group'] = gold_encoded['age_group'].map({
    'Age Group1': 1, 'Age Group2': 2, 'Age Group3': 3,
    'Age Group4': 4, 'Age Group5': 5, 'Age Group6': 6
})
gold_encoded['income_level'] = gold_encoded['income_level'].str.replace(
    'Level', '', regex=False).astype(int)
gold_encoded['household_size'] = gold_encoded['household_size'].map({
    '1': 1, '2': 2, '3': 3, '4': 4, '5+': 5
})
gold_encoded['kid_count'] = gold_encoded['kid_count'].map({
    'None/Unknown': 0, '1': 1, '2': 2, '3+': 3
})
gold_encoded['home_ownership'] = gold_encoded['home_ownership'].map({
    'Renter': 0, 'Probable Renter': 1, 'Unknown': 2,
    'Probable Owner': 3, 'Homeowner': 4
})
gold_encoded['marital_status'] = gold_encoded['marital_status'].map({
    'X': 0, 'Y': 1, 'Z': 2
})
gold_encoded['log_outcome'] = np.log1p(
    gold_encoded['avg_daily_campaign_spend']
)

w_cols = [
    'pre_avg_weekly_spend', 'pre_avg_weekly_trips',
    'pre_coupon_usage_rate', 'pre_loyalty_card_rate',
    'pre_department_breadth', 'pre_active_weeks',
    'clean_window_days'
]
x_cols = [
    'age_group', 'marital_status', 'income_level',
    'household_size', 'home_ownership', 'kid_count'
]

all_covariates = w_cols + x_cols

print(f"Shape: {gold_encoded.shape}")
print(f"Treatment balance — TypeA: {gold_encoded['treatment'].sum()}, "
      f"TypeBC: {(1-gold_encoded['treatment']).sum()}")

In [0]:
# Cell 2 — Build DoWhy CausalModel
# Encode the edges confirmed by PC algorithm + domain knowledge

causal_graph = """
digraph {
    treatment -> log_outcome;
    pre_avg_weekly_spend -> log_outcome;
    pre_loyalty_card_rate -> log_outcome;
    pre_loyalty_card_rate -> pre_avg_weekly_spend;
    pre_avg_weekly_spend -> pre_department_breadth;
    income_level -> pre_avg_weekly_spend;
    pre_loyalty_card_rate -> pre_avg_weekly_trips;
    pre_department_breadth -> pre_avg_weekly_trips;
    pre_active_weeks -> pre_avg_weekly_trips;
    pre_coupon_usage_rate -> pre_loyalty_card_rate;
    pre_coupon_usage_rate -> clean_window_days;
    age_group -> pre_loyalty_card_rate;
    pre_active_weeks -> pre_department_breadth;
    home_ownership -> age_group;
    kid_count -> age_group;
    marital_status -> household_size;
    home_ownership -> household_size;
    kid_count -> household_size;

    pre_avg_weekly_spend -> treatment;
    pre_avg_weekly_trips -> treatment;
    pre_coupon_usage_rate -> treatment;
    pre_loyalty_card_rate -> treatment;
    pre_department_breadth -> treatment;
    pre_active_weeks -> treatment;
    clean_window_days -> treatment;
    income_level -> treatment;
    age_group -> treatment;
}
"""

model = CausalModel(
    data=gold_encoded[['treatment', 'log_outcome'] + all_covariates],
    treatment='treatment',
    outcome='log_outcome',
    graph=causal_graph
)

# Identify the estimand
identified_estimand = model.identify_effect(
    proceed_when_unidentifiable=True
)
print(identified_estimand)

In [0]:
# Cell 3 — Get base estimate from DoWhy
# Use linear regression estimator for refutation
estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
    confidence_intervals=True,
    test_significance=True
)

print(f"DoWhy ATE estimate: {estimate.value:.4f}")
print(estimate)

In [0]:
print("Backdoor variables:", identified_estimand.backdoor_variables)